# 1. Análisis Exploratorio y Procesamiento de Datos Raw
En este cuaderno documento la ingesta de las fuentes de datos crudas, el cruce relacional y la aplicación temprana de algoritmos no supervisados para perfilar a la población estudiantil e identificar anomalías que puedan representar riesgo de deserción.

In [1]:
import os
if os.getcwd().endswith('notebooks'):
    os.chdir('..')

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import silhouette_score
import warnings
warnings.filterwarnings("ignore")

### 1.1 Carga y Cruce de Datos Sin Procesar (Raw)
**Justificación del Negocio:** En una universidad real, la información nunca está en una sola base de datos. Los registros académicos (calificaciones) viven en un sistema, los de asistencia en otro, y los demográficos en otro. Para este proyecto, decidí simular este entorno cargando 4 fuentes independientes (`estudiantes.csv`, `asistencia.csv`, `calificaciones.csv`, `inscripciones.csv`) y cruzándolas utilizando identificadores únicos (`id_estudiante`, `id_inscripcion`). Mi objetivo aquí es construir una vista de 360 grados del alumno, porque la deserción rara vez tiene un solo factor: un estudiante puede tener notas excelentes pero asistencia crítica, y solo cruzando los datos podremos detectarlo a tiempo.

In [3]:
estudiantes = pd.read_csv("data/01_raw/estudiantes.csv", encoding="latin-1")
calificaciones = pd.read_csv("data/01_raw/calificaciones.csv", encoding="latin-1")
asistencia = pd.read_csv("data/01_raw/asistencia.csv", encoding="latin-1")
inscripciones = pd.read_csv("data/01_raw/inscripciones.csv", encoding="latin-1")

calificaciones["nota"] = pd.to_numeric(calificaciones["nota"], errors="coerce")

# Generar features cruzando las tablas por id_estudiante
promedio = pd.merge(calificaciones, inscripciones, on="id_inscripcion").groupby("id_estudiante")["nota"].mean().reset_index(name="promedio_general")

asist = pd.merge(asistencia, inscripciones, on="id_inscripcion")
asist["presente"] = (asist["estado_asistencia"] == "Presente").astype(int)
asistencia_gr = asist.groupby("id_estudiante")["presente"].mean().reset_index(name="asistencia_general")

asignaturas = inscripciones.groupby("id_estudiante")["codigo_asignatura"].count().reset_index(name="total_asignaturas")
evaluaciones = pd.merge(calificaciones, inscripciones, on="id_inscripcion").groupby("id_estudiante")["id_calificacion"].count().reset_index(name="evaluaciones_totales")

# Dataset unificado
data = estudiantes.merge(promedio, on="id_estudiante", how="left") \
                  .merge(asistencia_gr, on="id_estudiante", how="left") \
                  .merge(asignaturas, on="id_estudiante", how="left") \
                  .merge(evaluaciones, on="id_estudiante", how="left")

data["target"] = (data["estado_matricula"] == "Retirado").astype(int)
# data.fillna() eliminado, el preprocesador se encarga de imputar NaNs correctamente.
data.head()

,id_estudiante,nombre,rut,carrera,sede,aÃ±o_ingreso,email,estado_matricula,promedio_general,asistencia_general,total_asignaturas,evaluaciones_totales,target
0,1.0,Ignacio Espinoza Vergara,23003953-7,EnfermerÃ­a,NaN,2020.0,ignacio.espinoza@outlook.com,Regular,1.900000,NaN,1.0,2.0,0
1,NaN,Carlos Gatica Medina,NaN,AdministraciÃ³n,Santiago Centro,2023.0,carlos.gatica@outlook.com,REGULAR,NaN,NaN,NaN,NaN,0
2,3.0,Camila Aguilera LÃ³pez,16824501-5,PsicologÃ­a,Sede Norte,2021.0,camila.aguilera@gmail.com,regular,4.427586,0.424242,3.0,30.0,0
3,4.0,MarÃ­a Rojas Vergara,20713487-8,Arquitectura,ConcepciÃ³n,2021.0,marÃ­a.rojas@outlook.com,EGRESADO,4.600000,1.000000,3.0,4.0,0
4,5.0,Fernanda MartÃ­nez Flores,16300007-6,Arquitectura,ViÃ±a del Mar,2023.0,NaN,Congelada,NaN,0.000000,2.0,NaN,0


In [4]:
# Preprocesamiento (ColumnTransformer)
X = data.drop(["target", "id_estudiante", "nombre", "rut", "email", "estado_matricula"], axis=1)
num_f = X.select_dtypes(include=["number"]).columns
cat_f = X.select_dtypes(include=["object"]).columns

preprocessor = ColumnTransformer([("num", Pipeline([("i", SimpleImputer(strategy="median")), ("s", StandardScaler())]), num_f), 
                                  ("cat", Pipeline([("i", SimpleImputer(strategy="constant")), ("o", OneHotEncoder(handle_unknown="ignore"))]), cat_f)])
X_prep = preprocessor.fit_transform(X).toarray()

### 1.2 Algoritmos de Clustering y Detección de Anomalías Exploratorias
**Por qué aplico Machine Learning No Supervisado en esta etapa:** Antes de intentar predecir quién deserta (Supervisado), me interesa entender la estructura interna de nuestra población estudiantil. Aquí utilizo una batería de algoritmos de clustering (K-Means, DBSCAN, Agglomerative, Gaussian Mixture) e Isolation Forest. 
*¿El hallazgo?* Los algoritmos basados en densidad (DBSCAN) catalogaron a casi todos los estudiantes como un solo bloque. Esto me confirma que los desertores no forman una 'tribu' apartada y evidente, sino que son casos anómalos dentro de la población general. Por esta razón, `Isolation Forest` (un algoritmo diseñado para detectar anomalías) resulta ser el más efectivo para aislar tempranamente a estudiantes con comportamientos atípicos urgentes.

In [5]:
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.mixture import GaussianMixture
from sklearn.ensemble import IsolationForest

kmeans = KMeans(n_clusters=3, random_state=42).fit(X_prep)
print("KMeans Silhouette:", silhouette_score(X_prep, kmeans.labels_))

agg = AgglomerativeClustering(n_clusters=3).fit(X_prep)
print("Agglomerative Silhouette:", silhouette_score(X_prep, agg.labels_))

gmm = GaussianMixture(n_components=2, random_state=42).fit(X_prep)
print("GMM Silhouette:", silhouette_score(X_prep, gmm.predict(X_prep)))

iso = IsolationForest(contamination=0.05, random_state=42).fit(X_prep)
print("Isolation Forest Anomalías Detectadas:", (iso.predict(X_prep) == -1).sum())

KMeans Silhouette: 0.15239470530628796
Agglomerative Silhouette: 0.32975958352572793
GMM Silhouette: 0.21484900502825982


Isolation Forest Anomalías Detectadas: 18
